In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
from football_analytics.analyses.passing.features import passing_feature_columns
import os
os.environ["OMP_NUM_THREADS"] = "4"
from sklearn.preprocessing import StandardScaler


In [ ]:
df_2018_wc = pd.read_parquet("../../../artifacts/features/player_level_wc2018.parquet")
df_2022_wc = pd.read_parquet("../../../artifacts/features/player_level_wc2022.parquet")
df_2024_euro = pd.read_parquet("../../../artifacts/features/player_level_euro2024.parquet")
df_2020_euro = pd.read_parquet("../../../artifacts/features/player_level_euro2020.parquet")
df_2023_afcon = pd.read_parquet("../../../artifacts/features/player_level_afcon2023.parquet")
df_2024_copa = pd.read_parquet("../../../artifacts/features/player_level_copa2024.parquet")

df_2018_wc["dataset"] = "WC2018"
df_2022_wc["dataset"] = "WC2022"
df_2024_euro["dataset"] = "EURO2024"
df_2020_euro["dataset"] = "EURO2020"
df_2023_afcon["dataset"] = "AFCON2023"
df_2024_copa["dataest"] = "COPA2024"

df = pd.concat([df_2018_wc, df_2022_wc, df_2024_euro, df_2020_euro, df_2024_copa, df_2023_afcon], ignore_index=True)
df = df[df["player_position"] != "Goalkeeper"]
passing_cols = passing_feature_columns()
passing_df = df[["player_key", "player_position", "player"] + passing_cols]
orig_df = df.copy()

df = passing_df.copy()
df = df.dropna(subset=["player_position"])

FEATURES = [
    'passes_per_90','short_passes_per_90','medium_passes_per_90','long_passes_per_90',
    'crosses_per90','switches_per90','throughballs_per90','cutbacks_per90','backheels_per90',
    'passes_under_pressure_per90','key_passes_per90','assists_per90','pct_passes_under_pressure',
    'avg_pass_length','std_pass_length','pct_short_pass','pct_med_pass','pct_long_pass',
    'pct_ground_pass','pct_low_pass','pct_high_pass','pct_left_foot_pass','pct_right_foot_pass',
    'pct_head_pass','pct_foot_pass','pct_other_pass','pct_keeper_arm_pass',
    'pct_progressive_passes','pct_lateral_passes','pct_defensive_passes',
    'pct_pass_from_def_third','pct_pass_from_mid_third','pct_pass_from_att_third',
    'pct_pass_from_left_channel','pct_pass_from_central_channel','pct_pass_from_right_channel',
    'pct_pass_to_def_third','pct_pass_to_mid_third','pct_pass_to_left_channel',
    'pct_pass_to_central_channel','pct_pass_to_right_channel',
    'pct_passes_final_third','pct_passes_into_box',
    'pct_pass_from_zone_dl','pct_pass_to_zone_dl','pct_pass_from_zone_dc','pct_pass_to_zone_dc',
    'pct_pass_from_zone_dr','pct_pass_to_zone_dr','pct_pass_from_zone_ml','pct_pass_to_zone_ml',
    'pct_pass_from_zone_mc','pct_pass_to_zone_mc','pct_pass_from_zone_mr','pct_pass_to_zone_mr',
    'pct_pass_from_zone_al','pct_pass_to_zone_al','pct_pass_from_zone_ac','pct_pass_to_zone_ac',
    'pct_pass_from_zone_ar','pct_pass_to_zone_ar',
    'pct_pass_def_to_mid','pct_pass_def_to_att','pct_pass_mid_to_att','pct_pass_mid_to_mid',
    'pct_pass_att_to_mid','pct_pass_def_to_def','pct_pass_att_to_att',
    'pct_pass_left_to_centre','pct_pass_left_to_right','pct_pass_right_to_centre',
    'pct_pass_right_to_left','pct_pass_centre_to_left','pct_pass_centre_to_right',
    'pct_pass_wide_to_box','pct_pass_centre_to_box','pct_pass_def_to_box',
    'ttl_passes_F','pct_passes_F','passes_F_per90',
    'ttl_passes_FR','pct_passes_FR','passes_FR_per90',
    'ttl_passes_R','pct_passes_R','passes_R_per90',
    'ttl_passes_BR','pct_passes_BR','passes_BR_per90',
    'ttl_passes_B','pct_passes_B','passes_B_per90',
    'ttl_passes_BL','pct_passes_BL','passes_BL_per90',
    'ttl_passes_L','pct_passes_L','passes_L_per90',
    'ttl_passes_FL','pct_passes_FL','passes_FL_per90',
    'pass_angle_mean_overall','pass_angle_var_overall',
    'pass_angle_mean_def_third','pass_angle_var_def_third',
    'pass_angle_mean_mid_third','pass_angle_var_mid_third',
    'pass_angle_mean_att_third','pass_angle_var_att_third',
    'pass_angle_mean_left_channel','pass_angle_var_left_channel',
    'pass_angle_mean_centre_channel','pass_angle_var_centre_channel',
    'pass_angle_mean_right_channel','pass_angle_var_right_channel'
]

df0 = passing_df.copy()

df1 = df0[
    (df0["has_minutes"] == 1) &
    (df0["has_pass_events"] == 1) &
    (df0["passes_per_90"].fillna(0) > 0) &
    (df0["player_position"].notna())
].copy()

In [ ]:
len(df0), len(df1), df1["player_position"].value_counts().head(30)

In [ ]:
df1 = df1[df1['player_position'] == 'Left Center Back']
df1

In [ ]:


X = df1[FEATURES].copy()
X[FEATURES].isna().mean().sort_values(ascending=False).head(15)
X["backheels_per90"] = X["backheels_per90"].fillna(0.0)

df_model = X

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled = scaler.fit_transform(df_model[FEATURES])
X_scaled = pd.DataFrame(X_scaled, columns=FEATURES, index=df_model.index)
X_scaled.head(10)

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled = scaler.fit_transform(df_model[FEATURES])
X_scaled = pd.DataFrame(X_scaled, columns=FEATURES, index=df_model.index)
X_scaled.head(10)

from sklearn.decomposition import PCA

pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_scaled)
X_pca = pd.DataFrame(
    X_pca,
    index=df_model.index,
    columns=[f"PC{i+1}" for i in range(X_pca.shape[1])]
)

import numpy as np
import matplotlib.pyplot as plt

plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.show()

threshold = 0.90  # For 90% explained variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(cumulative_variance >= threshold) + 1

print(f"Number of components that explain {threshold*100}% variance: {n_components}")

In [ ]:
pca = PCA(n_components=24)
X_pca=pca.fit_transform(X_scaled)
X_pca = pd.DataFrame(
    X_pca,
    index=df_model.index,
    columns=[f"PC{i+1}" for i in range(X_pca.shape[1])]
)


In [ ]:
from sklearn.cluster import KMeans

wcss = []
for i in range(1,40):
    kmeans_pca = KMeans(n_clusters = i, init = "k-means++", random_state = 42)
    kmeans_pca.fit(X_pca)
    wcss.append(kmeans_pca.inertia_)

plt.figure(figsize = (10,8))
plt.plot(range(1,40), wcss, marker = 'o', linestyle = '--')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('K-means with PCA clustering')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=8, random_state=42)
df_model["role_id"] = kmeans.fit_predict(X_pca.iloc[:, :30])
df_model["role_id"].value_counts()

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_std = pd.DataFrame(
    scaler.fit_transform(df_model[FEATURES]),
    columns=FEATURES,
    index=df_model.index
)

X_std["role_id"] = df_model["role_id"]
X_std.sample(10)

role_profiles = X_std.groupby("role_id")[FEATURES].mean()
for r in role_profiles.index:
    print(f"\nROLE {r}")
    print("Top + traits")
    print(role_profiles.loc[r].sort_values(ascending=False).head(10))
    print("\nTop - traits")
    print(role_profiles.loc[r].sort_values().head(10))